In [0]:
CREATE DATABASE SALES

In [0]:
CREATE TABLE sales.orders
(
    order_id INT,
    order_date DATE,
    customer_id INT,
    customer_name STRING,
    customer_email STRING,
    product_id INT,
    product_name STRING,
    product_category STRING,
    regionid INT,
    regionName STRING,
    country STRING,
    quantity INT,
    unitprice DECIMAL(10,2),
    totalamount DECIMAL(10,2) 
)

In [0]:
INSERT INTO sales.orders
VALUES
(1, '20234-02-01',101,'Alice Johnson','alice@gmail.com',201,'Laptop','Electronics',301,'North America','USA',2,800.00,1600.00),
(2, '2023-02-02',102,'Bob Brown','bob@yahoo.com',202,'Monitor','Electronics',302,'North America','Canada',1,500.00,500.00),
(3, '2023-02-03',103,'Charlie Lee','charlie@hotmail.com',203,'Keyboard','Electronics',303,'Europe','UK',3,100.00,300.00),
(4, '2023-02-04',104,'David Kim','david@gmail.com',204,'Headphones','Electronics',304,'Europe','Germany',4,150.00,600.00),
(5, '2023-02-05',105,'Eve Chen','eve@yahoo.com',205,'Mouse','Electronics',305,'Asia','China',5,75.00,375.00),
(6, '2023-02-06',106,'Frank Nguyen','frank@hotmail.com',206,'Laptop','Electronics',306,'Asia','Japan',2,800.00,1600.00),
(7, '2023-02-07',107,'Grace Park','grace@gmail.com',207,'Keyboard','Electronics',307,'Asia','India',3,100.00,300.00),
(8, '2023-02-08',108,'Henry Lee','henry@yahoo.com',208,'Monitor','Electronics',308,'Asia','China',1,500.00,500.00),
(9, '2023-02-09',109,'Ivy Kim','ivy@hotmail.com',209,'Headphones','Electronics',309,'Europe','France',4,150.00,600.00),
(10, '2023-02-10',110,'Jack Chen','jack@gmail.com',210,'Mouse','Electronics',310,'Europe','Germany',5,75.00,375.00)

#### Incremental Load Process:

In [0]:
-- CREATE DATA WAREHOUSE DATABASE:
CREATE DATABASE salesDWH;

In [0]:
-- Insert records for incremental load:
INSERT INTO sales.orders
VALUES
(11, '2023-02-11',111,'Ivy Kim','ivy@hotmail.com',211,'Headphones','Electronics',311,'Europe','France',4,150.00,600.00),
(12, '2023-02-12',112,'Jack Chen','jack@gmail.com',212,'Mouse','Electronics',312,'Europe','Germany',5,75.00,375.00),
(13, '2023-02-13',113,'Grace Park','grace@gmail.com',213,'Keyboard','Electronics',313,'Asia','India',3,100.00,300.00),
(14, '2023-02-14',114,'Henry Lee','henry@yahoo.com',214,'Monitor','Electronics',314,'Asia','China',1,500.00,500.00),    
(15, '2023-02-15',115,'Frank Nguyen','frank@hotmail.com',215,'Laptop','Electronics',315,'Asia','Japan',2,800.00,1600.00)

##### Source Layer (Database):

In [0]:
select * from sales.orders

##### Staging Layer:

In [0]:
-- Inserting into staging table using incremental load, here create and replace table will act as truncate
CREATE OR REPLACE TABLE salesdwh.stg_sales
AS
SELECT * FROM sales.orders
WHERE order_date > '2023-02-10'

##### Transformation:

In [0]:
-- Transformation on top os staging layer:
CREATE OR REPLACE VIEW salesdwh.trans_sales
AS 
SELECT * FROM salesdwh.stg_sales WHERE quantity IS NOT NULL

In [0]:
SELECT * FROM salesdwh.trans_sales

##### Core Layer:

In [0]:
-- Creating the curated/core layer
CREATE OR REPLACE TABLE salesdwh.core_sales
(
    order_id INT,
    order_date DATE,
    customer_id INT,
    customer_name STRING,
    customer_email STRING,
    product_id INT,
    product_name STRING,
    product_category STRING,
    regionid INT,
    regionName STRING,
    country STRING,
    quantity INT,
    unitprice DECIMAL(10,2),
    totalamount DECIMAL(10,2) 
)

In [0]:
-- Insert data into the core layer from trans layer:
INSERT INTO salesdwh.core_sales
SELECT * FROM salesdwh.trans_sales;

In [0]:
select * from salesdwh.core_sales